# DeepLoop2 Beta — Inference Walkthrough

This notebook provides a beginner-friendly walkthrough of the **DeepLoop2 beta inference workflow**.

DeepLoop2 enhances sparse chromatin contact maps using pretrained DenseUNet models. The public beta focuses on **inference**; model-training code is not included.

### What you will do

1. Set up the DeepLoop2 repository and Python environment.
2. Check whether a GPU is available.
3. Specify an anchor BED file, an anchor-to-anchor interaction file, and a pretrained checkpoint.
4. Inspect the expected input formats.
5. Run DeepLoop2 on one chromosome.
6. Inspect the prediction output.
7. Optionally export the result to Cooler format.
8. Learn how to compare candidate pretrained models before running a full dataset.

> **Beta note:** Public checkpoint URLs and downloadable example data are intentionally left configurable until the beta model set is finalized.

## 1. Get DeepLoop2

### Local Jupyter
If you already cloned the repository and launched Jupyter from the repository root, you can skip cloning and set `REPO_DIR` accordingly.

In [ ]:
!git clone https://github.com/dylan-plummer/DeepLoop2.0.git

from pathlib import Path

# For local use, change it to your cloned repository path.
repo_dir = Path("/content/DeepLoop2.0")
print("Repository directory:", repo_dir)

## 2. Install/check dependencies

The final beta repository will include `requirements.txt`.

In [ ]:
!pip install -r /content/DeepLoop2.0/requirements.txt

import sys
import numpy as np
import pandas as pd
import torch

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Device:", "cuda" if torch.cuda.is_available() else "cpu")

## 3. Choose a chromosome and provide your files

DeepLoop2 currently expects:

- an **anchor BED reference** for the chromosome;
- a chromosome-level **anchor-to-anchor interaction table**;
- a pretrained DeepLoop2 **`.pt` checkpoint**.

Edit the paths below to point to your files.

In [ ]:
chromosome = "chr10"

# Input directory and filename
input_dir = Path("/content/data/input")
input_name = f"{chromosome}.anchor.to.anchor"
# Directory containing chr*.bed files, or change this to a direct BED path
anchor_dir = Path("/content/data/anchors")

# Pretrained model checkpoint
weights = repo_dir / "model" / "checkpoints" / "YOUR_MODEL.pt"

# Prediction output
output_dir = Path("/content/deeploop2_output")
output_dir.mkdir(parents=True, exist_ok=True)

print("Chromosome:", chromosome)
print("Interaction file:", input_dir / input_name)
print("Anchor reference:", anchor_dir)
print("Checkpoint:", weights)
print("Output directory:", output_dir)

## 4. Understand the input format

### Anchor BED
DeepLoop2 reads the first four tab-separated columns as:

```text
chrom    start    end    anchor_id
```

### Anchor-to-anchor interaction table
The default inference mode expects:

```text
anchor1    anchor2    obs    exp
```

and computes:

```text
ratio = (obs + 5) / (exp + 5)
```

The pseudocount can be changed with `--dummy`. If your file already contains a `ratio` column, `predict.py` can use that instead through `--val_cols`.

The cells below preview your files without modifying them.

In [ ]:
anchor_file = anchor_dir / f"{chromosome}.bed" if anchor_dir.is_dir() else anchor_dir
interaction_file = input_dir / input_name

if anchor_file.exists():
    anchors = pd.read_csv(
        anchor_file, sep="\t", header=None, usecols=[0, 1, 2, 3],
        names=["chrom", "start", "end", "anchor_id"]
    )
    display(anchors.head())
    print(f"Anchors: {len(anchors):,}")
else:
    print("Anchor file not found yet:", anchor_file)

if interaction_file.exists():
    interactions = pd.read_csv(
        interaction_file, sep="\t", header=None,
        names=["anchor1", "anchor2", "obs", "exp"]
    )
    display(interactions.head())
    print(f"Interactions: {len(interactions):,}")
else:
    print("Interaction file not found yet:", interaction_file)

## 5. Run DeepLoop2 inference

The command below calls the repository's `inference/predict.py` script. The current defaults use 128 × 128 tiles, a step size of 128, and a maximum diagonal distance of 384 pixels.

For your first test, we recommend running **one chromosome** before scaling to the rest of the genome.

In [ ]:
import subprocess

predict_script = repo_dir / "inference" / "predict.py"

cmd = [
    sys.executable, str(predict_script),
    "--full_matrix_dir", str(input_dir),
    "--input_name", input_name,
    "--weights", str(weights),
    "--out_dir", str(output_dir),
    "--anchors", str(anchor_file),
    "--chromosome", chromosome,
    "--small_matrix_size", "128",
    "--step_size", "128",
    "--max_dist", "384",
]
print("Command to run:")
print(" ".join(cmd))

required = [predict_script, interaction_file, anchor_file, weights]
missing = [str(p) for p in required if not p.exists()]

if missing:
    print("\nNot running yet. Please provide these files first:")
    for p in missing:
        print(" -", p)
else:
    subprocess.run(cmd, check=True)

## 6. Inspect the prediction

For a chromosome such as `chr10`, DeepLoop2 writes:

```text
chr10.denoised.anchor.to.anchor
```

with three tab-separated columns:

```text
anchor1    anchor2    denoised_value
```

In [ ]:
prediction_file = output_dir / f"{chromosome}.denoised.anchor.to.anchor"

if prediction_file.exists():
    prediction = pd.read_csv(
        prediction_file, sep="\t", header=None,
        names=["anchor1", "anchor2", "denoised"]
    )
    display(prediction.head())
    print(f"Predicted interactions: {len(prediction):,}")
    print(prediction["denoised"].describe())
else:
    print("Prediction output has not been generated yet:", prediction_file)

## 7. Optional: export to `.cool`

DeepLoop2 includes `utils/convert_to_cooler.py` for converting anchor-to-anchor interaction tables into Cooler format.

The utility supports anchor-resolution output, optional uniform binning, single-chromosome conversion, and optional `.mcool` generation.

Because the converter's column settings must match the exact interaction file being converted, first inspect its command-line options:

In [ ]:
converter = repo_dir / "utils" / "convert_to_cooler.py"

if converter.exists():
    subprocess.run([sys.executable, str(converter), "--help"], check=True)
else:
    print("Converter not found yet:", converter)

## 8. Choosing among pretrained models

If you do not know which checkpoint best matches your input depth, **do not immediately run every chromosome with one arbitrary model**.

For the beta workflow, we recommend:

1. choose one or a few representative chromosomes/loci;
2. run the same input with several candidate pretrained checkpoints;
3. compare the resulting contact maps;
4. look for improved structural signal while checking for obvious artifacts;
5. choose the most suitable checkpoint and then scale up inference.

The code below provides a simple pattern for collecting candidate checkpoint paths. Automated model-comparison visualization will be added once the public checkpoint set and comparison workflow are finalized.

In [ ]:
checkpoint_dir = repo_dir / "model" / "checkpoints"

if checkpoint_dir.exists():
    checkpoints = sorted(checkpoint_dir.glob("*.pt"))
    if checkpoints:
        print("Available checkpoints:")
        for ckpt in checkpoints:
            print(" -", ckpt.name)
    else:
        print("No .pt checkpoints found in", checkpoint_dir)
else:
    print("Checkpoint directory not found:", checkpoint_dir)

## 9. Scaling up

Once you have confirmed that:

- the input format is correct;
- the checkpoint loads successfully;
- one-chromosome output looks reasonable;
- you have selected an appropriate pretrained model;

use the same inference command for the remaining chromosomes in your dataset.

For large datasets or HPC environments, chromosome-level jobs can be submitted independently.

---

### Feedback

DeepLoop2 is currently a beta release. If you encounter an error, please report the command you ran, the relevant input format, the checkpoint used, and the full traceback in the repository issue tracker.